<a href="https://colab.research.google.com/github/stauntonjr/local_llm_notebooks/blob/master/Unsloth_Fine_Tuning_for_RNJ_1_3x_Faster_with_Packing_and_Better_RoPE_Kernels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

More details in this article: [Fine-Tuning RNJ-1 with Unsloth and Packed Training Samples](https://kaitchup.substack.com/p/fine-tuning-rnj-1-with-unsloth)

This notebook shows how to fine-tune LLMs, here, rnj-1, with Unsloth. It was made specifically to test the new packing feature that significantly accelerates fine-tuning.

I ran fine-tuning to teach machine translation, from English to Japanese and French, using the instruct and base versions of rnj-1.

# Installation

*Note: Unsloth has complicated installation instructions for Colab. In my case, a simple "pip install" worked.*

In [ ]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 4.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.2/375.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.3/289.3 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.

# Fine-Tuning Code

In [ ]:
from unsloth import FastLanguageModel,  UnslothTrainer, UnslothTrainingArguments
import torch, multiprocessing
from datasets import load_dataset
from transformers import set_seed, AutoTokenizer,DataCollatorForSeq2Seq
from trl import SFTTrainer, SFTConfig

set_seed(42)

iso_language = dict()
iso_language["en"] = "English"
iso_language["ja"] = "Japanese"
iso_language["fr"] = "French"


def FT(model_name, pair, explicit_packing=False):

  compute_dtype = torch.bfloat16

  bs = 16
  gas = 1
  mseqlen = 4096 #Maximum sequence length; reduce if you run out of memory

  lr = 2e-4

  output_dir = "./sft-rnj-1-lora/"

  model, tokenizer = FastLanguageModel.from_pretrained(
      model_name = model_name,
      fix_tokenizer=False,
      max_seq_length = mseqlen,
      dtype = compute_dtype,
      load_in_4bit=False
  )

  languages = pair.split("-")
  src_lang = languages[0]
  tgt_lang = languages[1]

  ds = load_dataset("Helsinki-NLP/opus-100", pair, split="train").train_test_split(test_size=0.001)
  ds_train = ds["train"]
  ds_test = ds["test"]

  def process(row):

      source = row['translation'][src_lang]
      target = row['translation'][tgt_lang]
      row["messages"] = [
          {"role": "system", "content": "You are a professional translator that translates messages from "+iso_language[src_lang]+" to "+iso_language[tgt_lang]+"."},
          {"role": "user", "content": source},
          {"role": "assistant", "content": target},
      ]
      row["text"] = tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=False)
      return row

  ds_train = ds_train.map(
      process,
      num_proc= multiprocessing.cpu_count(),
      load_from_cache_file=False,
  )
  print(ds_train[0]['text'])
  ds_train = ds_train.remove_columns(["messages"])

  ds_test = ds_test.map(
      process,
      num_proc= multiprocessing.cpu_count(),
      load_from_cache_file=False,
  )
  ds_test = ds_test.remove_columns(["messages"])

  model = FastLanguageModel.get_peft_model(
      model,
      r = 32,
      target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj","embed_tokens", "lm_head"],
      lora_alpha = 32,
      use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
      random_state = 3407,
  )

  training_arguments = UnslothTrainingArguments(
          output_dir=output_dir,
          optim="adamw_8bit",
          per_device_train_batch_size=bs,
          gradient_accumulation_steps=gas,
          log_level="debug",
          #save_strategy="epoch",
          save_steps=1000,
          logging_steps=25,
          learning_rate = lr,
          embedding_learning_rate = lr/10,
          bf16 = True,
          #num_train_epochs=1,
          max_steps=8000,
          warmup_ratio=0.1,
          report_to = "none",
          lr_scheduler_type="linear",
          max_length=mseqlen,
          dataset_text_field='text',
          dataset_num_proc=multiprocessing.cpu_count(),
          do_eval=True,
          per_device_eval_batch_size=bs,
          eval_steps=100,
          packing=explicit_packing,
          eval_strategy="steps",
  )

  trainer = UnslothTrainer(
      model = model,
      train_dataset=ds_train,
      eval_dataset=ds_test,
      processing_class=tokenizer,
      args = training_arguments
  )

  trainer_ = trainer.train()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Fine-Tuning Runs

In [ ]:
FT("EssentialAI/rnj-1", "en-fr")

Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'extrapolation_factor', 'attn_factor'}
Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'extrapolation_factor', 'attn_factor'}


==((====))==  Unsloth 2025.12.5: Fast Gemma3 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'extrapolation_factor', 'attn_factor'}
Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'extrapolation_factor', 'attn_factor'}


Unsloth: Gemma3 does not support SDPA - switching to fast eager.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

...

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Currently training with a batch size of: 16
The following columns in the Training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: attention_mask, translation, text. If attention_mask, translation, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
skipped Gemma3TextScaledWordEmbedding(128256, 4096, padding_idx=0): 501.0M params
bitsandbytes: will optimize Gemma3TextScaledWordEmbedding(128256, 4096, padding_idx=0) in fp32
skipped: 501.0M params
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 999,000 | Num Epochs = 1 | Total steps = 8,000
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 94,412,800 of 8,409,149,440 (1.12% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,1.385400,3.391976
200,1.089100,3.278517
300,1.070900,3.147907
400,1.088300,3.112308
500,1.116500,3.073731
600,1.088800,3.177827
700,1.105500,2.998948
800,1.112200,2.974522
900,1.075800,3.032098
1000,1.084700,2.987082


The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: attention_mask, translation, text. If attention_mask, translation, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 16
The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: attention_mask, translation, text. If attention_mask, translation, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 16
The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: attention_mask, translation, text. If attention_mask, translation, text are not expected by `PeftModelForCausalLM.forward`,

In [ ]:
FT("EssentialAI/rnj-1", "en-ja")

Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor', 'extrapolation_factor'}
Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor', 'extrapolation_factor'}


==((====))==  Unsloth 2025.12.5: Fast Gemma3 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor', 'extrapolation_factor'}


Unsloth: Gemma3 does not support SDPA - switching to fast eager.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor', 'extrapolation_factor'}


...

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,1.438100,2.623481
200,0.997400,2.547057
300,0.987300,2.400388
400,0.944900,2.396499
500,0.998000,2.305738
600,0.976200,2.391247
700,0.972100,2.389784
800,0.934900,2.307009
900,0.961900,2.295843
1000,0.898500,2.197091


The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: attention_mask, translation, text. If attention_mask, translation, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 16
The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: attention_mask, translation, text. If attention_mask, translation, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 16
The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: attention_mask, translation, text. If attention_mask, translation, text are not expected by `PeftModelForCausalLM.forward`,

Step,Training Loss,Validation Loss
100,1.438100,2.623481
200,0.997400,2.547057
300,0.987300,2.400388
400,0.944900,2.396499
500,0.998000,2.305738
600,0.976200,2.391247
700,0.972100,2.389784
800,0.934900,2.307009
900,0.961900,2.295843
1000,0.898500,2.197091


The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: attention_mask, translation, text. If attention_mask, translation, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 16
The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: attention_mask, translation, text. If attention_mask, translation, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 16
The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: attention_mask, translation, text. If attention_mask, translation, text are not expected by `PeftModelForCausalLM.forward`,

In [ ]:
FT("EssentialAI/rnj-1-instruct", "en-fr")

Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'extrapolation_factor', 'attn_factor'}
Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'extrapolation_factor', 'attn_factor'}


==((====))==  Unsloth 2025.12.5: Fast Gemma3 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'extrapolation_factor', 'attn_factor'}
Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'extrapolation_factor', 'attn_factor'}


Unsloth: Gemma3 does not support SDPA - switching to fast eager.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Currently training with a batch size of: 16
The following columns in the Training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: attention_mask, translation, text. If attention_mask, translation, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
skipped Gemma3TextScaledWordEmbedding(128256, 4096, padding_idx=0): 501.0M params
bitsandbytes: will optimize Gemma3TextScaledWordEmbedding(128256, 4096, padding_idx=0) in fp32
skipped: 501.0M params
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 999,000 | Num Epochs = 1 | Total steps = 8,000
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 94,412,800 of 8,409,149,440 (1.12% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,1.399100,3.495572
200,1.098600,3.418429
300,1.077300,3.280976
400,1.098100,3.251070
500,1.126900,3.217583
600,1.097000,3.220182
700,1.114200,3.055839
800,1.120000,3.112971
900,1.083600,3.150569
1000,1.095000,3.020606


The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: attention_mask, translation, text. If attention_mask, translation, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 16
The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: attention_mask, translation, text. If attention_mask, translation, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 16
The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: attention_mask, translation, text. If attention_mask, translation, text are not expected by `PeftModelForCausalLM.forward`,

In [ ]:
FT("EssentialAI/rnj-1-instruct", "en-ja")

Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'extrapolation_factor', 'attn_factor'}
Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'extrapolation_factor', 'attn_factor'}


==((====))==  Unsloth 2025.12.5: Fast Gemma3 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'extrapolation_factor', 'attn_factor'}
Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'extrapolation_factor', 'attn_factor'}


Unsloth: Gemma3 does not support SDPA - switching to fast eager.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

...

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Currently training with a batch size of: 16
The following columns in the Training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, translation, attention_mask. If text, translation, attention_mask are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
skipped Gemma3TextScaledWordEmbedding(128256, 4096, padding_idx=0): 501.0M params
bitsandbytes: will optimize Gemma3TextScaledWordEmbedding(128256, 4096, padding_idx=0) in fp32
skipped: 501.0M params
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 999,000 | Num Epochs = 1 | Total steps = 8,000
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 94,412,800 of 8,409,149,440 (1.12% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,1.437200,2.475286
200,1.003900,2.332254
300,0.995800,2.452955
400,0.948300,2.527883
500,1.005600,2.324945
600,0.983300,2.423063
700,0.979300,2.351043
800,0.943800,2.385298
900,0.972000,2.348791
1000,0.904700,2.354889


The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, translation, attention_mask. If text, translation, attention_mask are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 16
The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, translation, attention_mask. If text, translation, attention_mask are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 16
The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, translation, attention_mask. If text, translation, attention_mask are not expected by `PeftModelForCausalLM.forward`,

# Merge the Adapter

Some inference engines like vLLM don't support adapters with full modules like token embeddings and language modeling head. To use them, merging the adapter into the model is necessary.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_id    = "EssentialAI/rnj-1-instruct"   # your base model
adapter_id = "./sft-rnj-1-instruct-lora/checkpoint-8000/"      # your PEFT adapter (LoRA, etc.)
out_dir    = "./sft-rnj-1-instruct-lora-enfr"

tok = AutoTokenizer.from_pretrained(base_id)
base = AutoModelForCausalLM.from_pretrained(base_id, torch_dtype=torch.bfloat16, device_map="auto")

model = PeftModel.from_pretrained(base, adapter_id)
model = model.merge_and_unload()          # merges adapter into base weights

model.save_pretrained(out_dir, safe_serialization=True)
tok.save_pretrained(out_dir)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--EssentialAI--rnj-1-instruct/snapshots/75d870a1ea6822d44153a2aff523fa31b6c5e3c4/config.json
Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'extrapolation_factor', 'attn_factor'}
Model config Gemma3TextConfig {
  "_sliding_window_pattern": 1,
  "architectures": [
    "Gemma3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "attn_logit_softcapping": null,
  "bos_token_id": 2,
  "cache_implementation": "hybrid",
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "final_logit_softcapping": 30.0,
  "head_dim": 128,
  "hidden_act": "gelu_pytorch_tanh",
  "hidden_activation": "gelu_pytorch_tanh",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 16384,
  "layer_type": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    

...

Instantiating Gemma3ForCausalLM model under default dtype torch.bfloat16.
Generate config GenerationConfig {
  "bos_token_id": 2,
  "cache_implementation": "hybrid",
  "eos_token_id": 1,
  "pad_token_id": 0
}



Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/226 [00:00<?, ?B/s]

loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--EssentialAI--rnj-1-instruct/snapshots/75d870a1ea6822d44153a2aff523fa31b6c5e3c4/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 128000,
  "cache_implementation": "hybrid",
  "do_sample": true,
  "eos_token_id": 128009,
  "pad_token_id": 128001,
  "temperature": 0.2
}

Could not locate the custom_generate/generate.py inside EssentialAI/rnj-1-instruct.
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:916: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['model.embed_tokens', 'lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:569: UserWarning: Model with 

('./sft-rnj-1-instruct-lora-enfr/tokenizer_config.json',
 './sft-rnj-1-instruct-lora-enfr/special_tokens_map.json',
 './sft-rnj-1-instruct-lora-enfr/chat_template.jinja',
 './sft-rnj-1-instruct-lora-enfr/tokenizer.json')